In [2]:
import numpy as np
def resol_sist_triang_inf(a, b):
    """
    Resolve um sistema de equações lineares Ax = b,
    onde A é uma matriz triangular inferior.
    Parâmetros:
    a : array_like
        Matriz dos coeficientes (triangular inferior).
    b : array_like
        Vetor dos termos independentes.
    """
    n = len(b)
    x = np.zeros(n)
    x[0] = b[0] / a[0, 0]
    for k in range(1, n):
        s = 0
        for j in range(k):
            s += a[k, j] * x[j]
        x[k] = (b[k] - s) / a[k, k]
    return x

# Exemplo de uso
A = np.array([[1, 0, 0],
              [2, 1, 0],
              [2, -3, 1]], dtype=float)

b = np.array([-3, -1, -23], dtype=float)
x = resol_sist_triang_inf(A, b)
print("Sol:", x)

Sol: [-3.  5. -2.]


In [3]:
import numpy as np

# Método da substituição regressiva
def resol_sist_triang_sup(a, b):
    """Resolve um sistema de equações lineares Ax = b
    Parâmetros:
    a : array_like
        Matriz dos coeficientes (triangular superior).
    b : array_like
        Vetor dos termos independentes.
    """
    n = len(b)
    x = np.zeros(len(b))
    x[n-1] = b[n-1] / a[n-1,n-1]

    for k in range(n-2, -1, -1):
        s = 0
        for j in range(k+1, n):
            s += a[k,j] * x[j]
        x[k] = (b[k] - s) / a[k,k]

    return x

#Exemplo de uso
A = np.array([[3, 2, 4],
              [0, 1/3, 2/3],
              [0, 0, 8]], dtype=float)

b = np.array([1, 5/3, 0], dtype=float)
x = resol_sist_triang_sup(A, b)
print("Sol:", x)

Sol: [-3.  5.  0.]


In [6]:
import numpy as np
import math

# tem que ser simétrica definida positiva
def cholesky_slide(A):
    """
    Fatoração de Cholesky (seguindo o slide)
    ----------
    A : ndarray (n x n)
        Matriz simétrica definida positiva.
    Retorno
    -------
    G : ndarray (n x n)
        Matriz triangular inferior tal que A = G @ G.T
    """
    A = np.array(A, dtype=float)
    n = A.shape[0]
    G = np.zeros_like(A)
    # Passo 1: g11
    G[0,0] = math.sqrt(A[0,0])
    # Passo 2: primeira coluna (i = 2..n no slide -> i=1..n-1 em Python)
    for i in range(1, n):
        G[i,0] = A[i,0] / G[0,0]
    # Passo 3: colunas k = 2..n-1  (slide) => k = 1..n-2 (Python)
    for k in range(1, n-1):
        # soma = sum_{j=1}^{k-1} g_{k,j}^2  (slide, j=1..k-1) -> j=0..k-1 em Python
        soma = 0.0
        for j in range(0, k):
            soma += G[k,j]**2
        r = A[k,k] - soma
        if r <= 0:
            raise ValueError(f"Matriz não é definida positiva (r={r} <= 0) na posição k={k}.")
        G[k,k] = math.sqrt(r)
        # Linhas i = k+1..n  -> i = k+1..n-1
        for i in range(k+1, n):
            soma = 0.0
            for j in range(0, k):
                soma += G[i,j] * G[k,j]
            G[i,k] = (A[i,k] - soma)/G[k,k]
    # Passo 4: última linha g_nn
    soma = 0.0
    for j in range(0, n-1):
        soma += G[n-1, j]**2
    r = A[n-1, n-1] - soma
    if r <= 0:
        raise ValueError(f"Matriz não é definida positiva (r={r} <= 0) na última etapa.")
    G[n-1, n-1] = math.sqrt(r)
    return G

# -------------------------------------------------
# Questão 3 (Cholesky)
# A e b conforme enunciado
# -------------------------------------------------
A = np.array([
    [ 25.,  5., 10.,  5.],
    [  5., 10.,  5.,  7.],
    [ 10.,  5., 105., 34.],
    [  5.,  7., 34., 30.]
], dtype=float)
print("Matriz A =\n", A)

b = np.array([  5.,  40., 165., 152.], dtype=float)
print("Vetor b =", b)
# (a) Fatoração de Cholesky A = G G^T
G = cholesky_slide(A)
print("3a)(matriz G e) G =\n", G)
print("3a) (matriz G transporta e) G^T =\n", G.T)
print("verificação: ||A - G G^T||_inf =", np.max(np.abs(A - G @ G.T)))

# (b) Resolver Gy = b (substituição progressiva)
y = resol_sist_triang_inf(G, b)
print("3b) y (solução de Gy=b) =", y)

# (c) Resolver G^T x = y (substituição regressiva)
x = resol_sist_triang_sup(G.T, y)
print("3c) x (solução final) =", x)

# Checagens explicadas (o que deve acontecer)
print("\nChecagens — o que verificar e o esperado")
tol = 1e-10

# 1) Fatoração: A ≈ G G^T
err_fact = np.max(np.abs(A - G @ G.T))
print("1) Fatoração A = G G^T:")
print("   ||A - G G^T||_inf =", err_fact, "(esperado: muito próximo de 0)")
print("   Interpretação: se for ~0 (<= 1e-10), a fatoração está correta.")

# 2) Sistema inferior: Gy ≈ b
res1 = np.max(np.abs(G @ y - b))
print("\n2) Substituição progressiva (Gy = b):")
print("   max|Gy - b| =", res1, "(esperado: ~0)")
print("   Gy =", G @ y)
print("   b  =", b)
print("   Interpretação: Gy e b devem coincidir (diferença só por arredondamento).")

# 3) Sistema superior: G^T x ≈ y
res2 = np.max(np.abs(G.T @ x - y))
print("\n3) Substituição regressiva (G^T x = y):")
print("   max|G^T x - y| =", res2, "(esperado: ~0)")
print("   G^T x =", G.T @ x)
print("   y     =", y)
print("   Interpretação: G^T x e y devem coincidir (diferença só por arredondamento).")

# 4) Solução final: Ax ≈ b
res3 = np.max(np.abs(A @ x - b))
print("\n4) Solução verificada (Ax = b):")
print("   max|Ax - b| =", res3, "(esperado: ~0)")
print("   Ax =", A @ x)
print("   b  =", b)
print("   Interpretação: Ax deve ser igual a b (erro numérico muito pequeno é normal).")

Matriz A =
 [[ 25.   5.  10.   5.]
 [  5.  10.   5.   7.]
 [ 10.   5. 105.  34.]
 [  5.   7.  34.  30.]]
Vetor b = [  5.  40. 165. 152.]
3a)(matriz G e) G =
 [[ 5.  0.  0.  0.]
 [ 1.  3.  0.  0.]
 [ 2.  1. 10.  0.]
 [ 1.  2.  3.  4.]]
3a) (matriz G transporta e) G^T =
 [[ 5.  1.  2.  1.]
 [ 0.  3.  1.  2.]
 [ 0.  0. 10.  3.]
 [ 0.  0.  0.  4.]]
verificação: ||A - G G^T||_inf = 0.0
3b) y (solução de Gy=b) = [ 1. 13. 15. 20.]
3c) x (solução final) = [-1.  1.  0.  5.]

Checagens — o que verificar e o esperado
1) Fatoração A = G G^T:
   ||A - G G^T||_inf = 0.0 (esperado: muito próximo de 0)
   Interpretação: se for ~0 (<= 1e-10), a fatoração está correta.

2) Substituição progressiva (Gy = b):
   max|Gy - b| = 0.0 (esperado: ~0)
   Gy = [  5.  40. 165. 152.]
   b  = [  5.  40. 165. 152.]
   Interpretação: Gy e b devem coincidir (diferença só por arredondamento).

3) Substituição regressiva (G^T x = y):
   max|G^T x - y| = 0.0 (esperado: ~0)
   G^T x = [ 1. 13. 15. 20.]
   y     = [ 1. 13. 